In [2]:
import pandas as pd

df = pd.read_csv("../data/raw_matches.csv")
print("Total matches:", df.shape[0])
df.head()

Total matches: 4745


,ODI Match No,Match ID,Match Name,Series ID,Series Name,Match Date,Match Format,Team1 ID,Team1 Name,Team1 Captain,...,Umpire 2,Match Referee,Toss Winner,Toss Winner Choice,Match Winner,Match Result Text,MOM Player,Team1 Playing 11,Team2 Playing 11,Debut Players
0,488,65425,Australia Vs New Zealand 4Th Match,60879.0,"Benson & Hedges World Series Cup Australia, Ne...",1988-01-07,ODI,2.0,Australia,1572.0,...,RC Bailhache,NaN,Australia,bat,Australia,Australia won by 6 runs,1795.0,"['1767', '1793', '1754', '1572', '1871', '1795...","['1550', '1863', '1861', '1669', '1698', '1846...",[]
1,492,65428,New Zealand Vs Sri Lanka 7Th Match,60879.0,"Benson & Hedges World Series Cup Australia, Ne...",1988-01-12,ODI,5.0,New Zealand,1698.0,...,SG Randell,NaN,Sri Lanka,bowl,Sri Lanka,Sri Lanka won by 4 wickets (with 21 balls rema...,1810.0,"['1777', '1550', '1698', '1669', '1861', '1846...","['1810', '1864', '1789', '1762', '1666', '1664...",[]
2,495,65431,Australia Vs New Zealand 10Th Match,60879.0,"Benson & Hedges World Series Cup Australia, Ne...",1988-01-17,ODI,5.0,New Zealand,1698.0,...,AR Crafter,NaN,Australia,bowl,Australia,Australia won by 5 wickets (with 64 balls rema...,1795.0,"['1550', '1863', '1698', '1669', '1407', '1790...","['1793', '1767', '1773', '1754', '1871', '1795...",['1790']
3,496,65432,Australia Vs Sri Lanka 11Th Match,60879.0,"Benson & Hedges World Series Cup Australia, Ne...",1988-01-19,ODI,8.0,Sri Lanka,1664.0,...,TA Prue,NaN,Australia,bowl,Australia,Australia won by 3 wickets (with 3 balls remai...,1871.0,"['1810', '1864', '1789', '1753', '1762', '1666...","['1767', '1793', '1754', '1871', '1795', '1572...",['1753']
4,508,64326,New Zealand Vs England 3Rd Odi,60882.0,England tour of New Zealand - 1988 (1987/88),1988-03-16,ODI,1.0,England,1543.0,...,SJ Woodward,NaN,New Zealand,bowl,New Zealand,New Zealand won by 7 wickets (with 21 balls re...,1550.0,"['1758', '1629', '1770', '1850', '1543', '1865...","['1550', '1883', '1863', '1882', '1777', '1918...",['1883']


In [3]:
df = df[df["Match Format"] == "ODI"]
df = df[df["Match Winner"].notna()]

print("Total ODI matches:", df.shape[0])

Total ODI matches: 4535


In [4]:
df["team1_win"] = (df["Match Winner"] == df["Team1 Name"]).astype(int)

df[["Team1 Name", "Team2 Name", "Match Winner", "team1_win"]].head()

,Team1 Name,Team2 Name,Match Winner,team1_win
0,Australia,New Zealand,Australia,1
1,New Zealand,Sri Lanka,Sri Lanka,0
2,New Zealand,Australia,Australia,0
3,Sri Lanka,Australia,Australia,0
4,England,New Zealand,New Zealand,0


In [5]:
df["toss_winner_team1"] = (df["Toss Winner"] == df["Team1 Name"]).astype(int)


In [7]:
df.columns


Index(['ODI Match No', 'Match ID', 'Match Name', 'Series ID', 'Series Name',
       'Match Date', 'Match Format', 'Team1 ID', 'Team1 Name', 'Team1 Captain',
       'Team1 Runs Scored', 'Team1 Wickets Fell', 'Team1 Extras Rec',
       'Team2 ID', 'Team2 Name', 'Team2 Captain', 'Team2 Runs Scored',
       'Team2 Wickets Fell', 'Team2 Extras Rec', 'Match Venue (Stadium)',
       'Match Venue (City)', 'Match Venue (Country)', 'Umpire 1', 'Umpire 2',
       'Match Referee', 'Toss Winner', 'Toss Winner Choice', 'Match Winner',
       'Match Result Text', 'MOM Player', 'Team1 Playing 11',
       'Team2 Playing 11', 'Debut Players', 'team1_win', 'toss_winner_team1'],
      dtype='object')

In [8]:
team_country = {
    "India": "India",
    "Australia": "Australia",
    "England": "England",
    "South Africa": "South Africa",
    "New Zealand": "New Zealand",
    "Pakistan": "Pakistan",
    "Sri Lanka": "Sri Lanka",
    "West Indies": "West Indies",
    "Bangladesh": "Bangladesh",
    "Zimbabwe": "Zimbabwe",
    "Afghanistan": "Afghanistan",
    "Ireland": "Ireland",
    "Netherlands": "Netherlands",
    "Scotland": "Scotland",
    "UAE": "UAE",
    # Add any other ODI teams present in your CSV
}

In [9]:
df["team1_country"] = df["Team1 Name"].map(team_country)

In [10]:
df["team1_home"] = (df["Match Venue (Country)"] == df["team1_country"]).astype(int)


In [11]:
df[["Team1 Name", "Match Venue (Country)", "team1_country", "team1_home"]].head(10)

,Team1 Name,Match Venue (Country),team1_country,team1_home
0,Australia,Australia,Australia,1
1,New Zealand,Australia,New Zealand,0
2,New Zealand,Australia,New Zealand,0
3,Sri Lanka,Australia,Sri Lanka,0
4,England,New Zealand,England,0
5,India,United Arab Emirates,India,0
6,Sri Lanka,Bangladesh,Sri Lanka,0
7,New Zealand,India,New Zealand,0
8,West Indies,Australia,West Indies,0
9,West Indies,Australia,West Indies,0


In [13]:
# Sort matches by date
df = df.sort_values("Match Date")

# Initialize dictionary to keep track of team wins
team_wins = {}
team_matches = {}

# Store win percentage before each match
team1_win_pct = []

for idx, row in df.iterrows():
    team1 = row["Team1 Name"]
    team2 = row["Team2 Name"]

    # Calculate previous win % for Team1
    wins = team_wins.get(team1, 0)
    matches = team_matches.get(team1, 0)
    pct = wins / matches if matches > 0 else 0.5  # Default 0.5 if no history
    team1_win_pct.append(pct)

    # Update stats after current match
    team_matches[team1] = matches + 1
    team_wins[team1] = wins + (row["team1_win"])
    
df["team1_win_pct"] = team1_win_pct


In [14]:
features = ["toss_winner_team1", "team1_home", "team1_win_pct"]
X = df[features]
y = df["team1_win"]

In [19]:
# Initialize dictionary for Team2 wins and matches
team2_wins = {}
team2_matches = {}
team2_win_pct = []

for idx, row in df.iterrows():
    team2 = row["Team2 Name"]

    # Previous win % for Team2
    wins = team2_wins.get(team2, 0)
    matches = team2_matches.get(team2, 0)
    pct = wins / matches if matches > 0 else 0.5
    team2_win_pct.append(pct)

    # Update after current match
    team2_matches[team2] = matches + 1
    team2_wins[team2] = wins + (1 - row["team1_win"])  # Team2 win = 1 - team1_win

df["team2_win_pct"] = team2_win_pct


In [21]:
# Initialize head-to-head dictionary
head2head = {}
head2head_pct = []

for idx, row in df.iterrows():
    team1 = row["Team1 Name"]
    team2 = row["Team2 Name"]
    match_key = tuple(sorted([team1, team2]))  # unordered pair

    wins = head2head.get(match_key, [0, 0])  # [team1_wins, team2_wins]
    
    # Calculate head-to-head win % for Team1
    total = wins[0] + wins[1]
    pct = wins[0] / total if total > 0 else 0.5
    head2head_pct.append(pct)

    # Update head-to-head after current match
    if row["team1_win"] == 1:
        wins[0] += 1
    else:
        wins[1] += 1
    head2head[match_key] = wins

df["head2head_team1"] = head2head_pct


In [22]:
# Track last 5 match outcomes for each team
recent_form = {}
recent_win_pct = []

for idx, row in df.iterrows():
    team1 = row["Team1 Name"]

    # Get last 5 results
    last_results = recent_form.get(team1, [])
    if len(last_results) > 0:
        pct = sum(last_results) / len(last_results)
    else:
        pct = 0.5
    recent_win_pct.append(pct)

    # Update list of last 5 results
    last_results.append(row["team1_win"])
    if len(last_results) > 5:
        last_results.pop(0)
    recent_form[team1] = last_results

df["team1_recent_form"] = recent_win_pct


In [25]:
# Features including new engineered features
features = [
    "toss_winner_team1", 
    "team1_home", 
    "team1_win_pct", 
    "team2_win_pct", 
    "head2head_team1", 
    "team1_recent_form"
]
X = df[features]
y = df["team1_win"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predict and evaluate
preds = model.predict(X_test)
print("Upgraded model accuracy:", accuracy_score(y_test, preds))


Upgraded model accuracy: 0.639470782800441


In [26]:
from sklearn.metrics import confusion_matrix, classification_report

# Predict on test set
preds = model.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, preds)
print("Confusion Matrix:\n", cm)

# Classification report
print("\nClassification Report:\n", classification_report(y_test, preds))

Confusion Matrix:
 [[330 124]
 [203 250]]

Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.73      0.67       454
           1       0.67      0.55      0.60       453

    accuracy                           0.64       907
   macro avg       0.64      0.64      0.64       907
weighted avg       0.64      0.64      0.64       907



In [27]:
import joblib

# Save model
joblib.dump(model, "odi_winner_model.pkl")

# Load model later
# loaded_model = joblib.load("odi_winner_model.pkl")

['odi_winner_model.pkl']

In [30]:
def predict_match(team1, team2, toss_winner, venue_country):
    """
    Predicts the winner of an ODI match (Team1 vs Team2)
    Returns:
        pred: 1 if Team1 predicted to win, 0 if Team2
        prob: probability of Team1 winning
    """

    # Toss feature
    toss_feat = int(toss_winner == team1)
    
    # Home feature
    team1_country = team_country.get(team1, "Unknown")
    home_feat = int(venue_country == team1_country)
    
    # Historical win % for Team1
    team1_pct = df[df["Team1 Name"]==team1]["team1_win_pct"].iloc[-1] if not df[df["Team1 Name"]==team1].empty else 0.5
    
    # Historical win % for Team2
    team2_pct = df[df["Team2 Name"]==team2]["team2_win_pct"].iloc[-1] if not df[df["Team2 Name"]==team2].empty else 0.5
    
    # Head-to-head Team1 win %
    match_key = tuple(sorted([team1, team2]))
    h2h = head2head.get(match_key, [0,0])
    h2h_pct = h2h[0]/(h2h[0]+h2h[1]) if sum(h2h) > 0 else 0.5
    
    # Team1 recent form (last 5 matches)
    recent = recent_form.get(team1, [0,0,0,0,0])
    recent_pct = sum(recent)/len(recent)
    
    # Create DataFrame for features
    features_arr = pd.DataFrame([[toss_feat, home_feat, team1_pct, team2_pct, h2h_pct, recent_pct]],
                                columns=features)
    
    # Predict
    pred = model.predict(features_arr)[0]
    prob = model.predict_proba(features_arr)[0][1]
    
    return pred, prob


In [31]:
pred, prob = predict_match("India", "Australia", toss_winner="India", venue_country="India")
print(f"Predicted winner: {'Team1 (India)' if pred==1 else 'Team2 (Australia)'}")
print(f"Probability Team1 wins: {prob:.2f}")



Predicted winner: Team1 (India)
Probability Team1 wins: 0.53
